# Misspecified PINN Spectral Analysis

Trains a PINN with a misspecified base ODE (exponential decay, no forcing) and
additional data observations.  The ODE residual `u(t;θ) = z'(t;θ) + α z(t;θ)`
must converge to the true multi-frequency forcing.  Spectral bias predicts
low-frequency components recover before high-frequency ones.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from argparse import Namespace
from scipy.integrate import solve_ivp

ROOT = Path().resolve()
while not (ROOT / "src").is_dir():
    if ROOT == ROOT.parent:
        raise RuntimeError(f"Cannot find project root from {Path().resolve()}")
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
print(f"ROOT = {ROOT}")

from src.PINNs import PINN, PINNConfig, TrainingConfig, TrainingFrame
from src.spectral_analysis import compute_fft

sns.set_theme(style="whitegrid")
torch.manual_seed(42)
np.random.seed(42)

FIGURES_DIR = Path().resolve() / "figures"
FIGURES_DIR.mkdir(exist_ok=True)
print(f"Figures -> {FIGURES_DIR}")

## Training loop (misspecified physics + data loss)

In [ ]:
def train_misspec_pinn(
    model, alpha, t_span, y0, t_obs_t, y_obs_t,
    *, n_iter=5_000, n_colloc=512, lr=1e-3, data_weight=50.0,
    lr_decay=0.9995, rec_frq=100, t_eval_t=None, device=None, verbose=True,
):
    """
    Physics loss  : mean || z'(t;theta) + alpha * z(t;theta) ||^2
    Data loss     : mean || z(t_obs;theta) - y_obs ||^2
    Total         : physics + data_weight * data
    Hard-IC reparametrisation: z(t;theta) = y0 + t * NN(t)  (t0 = 0 assumed).
    """
    if device is None:
        device = next(model.parameters()).device

    t0_val, T = t_span
    y0_t = torch.tensor([y0], dtype=torch.float32, device=device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=lr_decay)

    frames: list[TrainingFrame] = []
    model.train()

    for it in range(n_iter + 1):
        optimizer.zero_grad()

        t_col = (
            torch.rand(n_colloc, 1, device=device) * (T - t0_val) + t0_val
        ).requires_grad_(True)
        z_col   = y0_t + t_col * model(t_col)
        dz_dt   = torch.autograd.grad(z_col.sum(), t_col, create_graph=True)[0]
        phys_loss = ((dz_dt - (-alpha * z_col)) ** 2).mean()

        z_obs     = y0_t + t_obs_t * model(t_obs_t)
        data_loss = ((z_obs - y_obs_t) ** 2).mean()

        total = phys_loss + data_weight * data_loss
        total.backward()
        optimizer.step()
        scheduler.step()

        if it % rec_frq == 0:
            model.eval()
            pred = None
            if t_eval_t is not None:
                with torch.no_grad():
                    pred = (y0_t + t_eval_t * model(t_eval_t)).cpu().numpy()
            snapshot = {k: v.detach().cpu().clone()
                        for k, v in model.state_dict().items()}
            frames.append(
                TrainingFrame(it, pred, total.item(),
                              phys_loss.item(), data_loss.item(), snapshot)
            )
            if verbose and it % max(1, n_iter // 5) == 0:
                print(
                    f"  iter {it:6d} | total {total.item():.3e}"
                    f" | phys {phys_loss.item():.3e}"
                    f" | data {data_loss.item():.3e}"
                )
            model.train()

    model.eval()
    return frames

## Configuration

In [ ]:
opt = Namespace()

opt.alpha  = 1.0
opt.freqs  = [1.0, 3.0, 5.0, 10.0]
opt.amps   = [1.0, 1.0, 1.0, 1.0]
opt.phases = [0.0, 0.0, 0.0, 0.0]
opt.y0     = [0.0]

opt.T        = 2.0
opt.N_POINTS = 200
opt.OBS_STEP = 4

if torch.cuda.is_available():
    opt.DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    opt.DEVICE = torch.device("mps")
else:
    opt.DEVICE = torch.device("cpu")
print(f"Device: {opt.DEVICE}")

model_cfg = PINNConfig(n_vars=1, width=128, depth=4)

tr_cfg = TrainingConfig(
    n_iter=20_000,
    n_colloc=512,
    lr=1e-3,
    rec_frq=50,
    save_snapshots=True,
    verbose=True,
)

t_eval      = np.linspace(0, opt.T, opt.N_POINTS)
sample_rate = opt.N_POINTS / opt.T

## Data generation

In [ ]:
u_true = np.array([
    sum(A * np.sin(2 * np.pi * f * t + ph)
        for f, A, ph in zip(opt.freqs, opt.amps, opt.phases))
    for t in t_eval
])

def rhs_true(t, z):
    forcing = sum(
        A * np.sin(2 * np.pi * f * t + ph)
        for f, A, ph in zip(opt.freqs, opt.amps, opt.phases)
    )
    return [-opt.alpha * z[0] + forcing]

print("Reference (RK45)...")
sol   = solve_ivp(rhs_true, [0, opt.T], opt.y0, t_eval=t_eval,
                  method="RK45", rtol=1e-9, atol=1e-11)
y_ref = sol.y.T

obs_idx  = np.arange(0, opt.N_POINTS, opt.OBS_STEP)
t_obs    = t_eval[obs_idx]
y_obs    = y_ref[obs_idx]

t_obs_t  = torch.tensor(t_obs,  dtype=torch.float32, device=opt.DEVICE).view(-1, 1)
y_obs_t  = torch.tensor(y_obs,  dtype=torch.float32, device=opt.DEVICE)
t_eval_t = torch.tensor(t_eval, dtype=torch.float32, device=opt.DEVICE).view(-1, 1)

## Training

In [ ]:
print(f"Training misspecified PINN ({tr_cfg.n_iter} iters)...")
model  = PINN(model_cfg).to(opt.DEVICE)
frames = train_misspec_pinn(
    model, opt.alpha,
    t_span=(0.0, opt.T),
    y0=opt.y0,
    t_obs_t=t_obs_t,
    y_obs_t=y_obs_t,
    n_iter=tr_cfg.n_iter,
    n_colloc=tr_cfg.n_colloc,
    lr=tr_cfg.lr,
    data_weight=1e6,
    rec_frq=tr_cfg.rec_frq,
    t_eval_t=t_eval_t,
    device=opt.DEVICE,
    verbose=tr_cfg.verbose,
)

## Residuals and plot

In [ ]:
# Compute residuals from training snapshots
rhs_misspec_torch = lambda t, z: -opt.alpha * z
residuals = model.compute_residuals(
    frames, rhs_misspec_torch, t_eval, opt.y0,
    hard_ic=True, device=opt.DEVICE,
)

# Misspecified PINN - four-panel residual spectral dynamics
valid       = [(f.iter_num, r) for f, r in zip(frames, residuals) if r is not None]
iters_m     = np.asarray([v[0] for v in valid])
res_list    = [v[1] for v in valid]
palette     = sns.color_palette("husl", len(opt.freqs))
snap_idx    = [0, len(res_list) // 2, len(res_list) - 1]
snap_colors = sns.color_palette("Blues_d", len(snap_idx))

frqs_true, spec_true = compute_fft(u_true, sample_rate)
true_amp = np.array([
    2.0 * float(spec_true[np.argmin(np.abs(frqs_true - f))]) for f in opt.freqs
])

norm_curves = {f: [] for f in opt.freqs}
for r in res_list:
    frqs_k, spec_k = compute_fft(r[:, 0], sample_rate)
    for f, a_true in zip(opt.freqs, true_amp):
        amp = 2.0 * float(spec_k[np.argmin(np.abs(frqs_k - f))])
        norm_curves[f].append(amp / a_true if a_true > 1e-12 else 0.0)

frqs_fin, spec_fin = compute_fft(res_list[-1][:, 0], sample_rate)
mask = frqs_fin <= max(opt.freqs) * 1.5

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle("Misspecified PINN - Residual Spectral Dynamics", fontsize=13)

ax = axes[0, 0]
ax.plot(t_eval, y_ref[:, 0], "k-", linewidth=1.4, label="True y(t)")
ax.plot(t_eval, [f.prediction for f in frames][-1][:, 0],
        color="steelblue", linewidth=1.2, alpha=0.9, label="PINN fit (final)")
ax.scatter(t_obs, y_obs[:, 0], s=8, c="gray", alpha=0.6, zorder=3, label="Observations")
ax.set_xlabel("t")
ax.set_ylabel("z(t)")
ax.set_title("State trajectory")
ax.legend(fontsize=8)

ax = axes[0, 1]
for f, col in zip(opt.freqs, palette):
    ax.plot(iters_m, norm_curves[f], color=col, linewidth=1.2, label=f"{f} Hz")
ax.axhline(1.0, color="gray", linestyle="--", linewidth=0.8, alpha=0.7)
ax.set_xlabel("Training iteration")
ax.set_ylabel("|FFT(u)| / |u_true|  at forcing freq")
ax.set_title("Spectral dynamics of residual u(t;theta)\n"
             "(dashed = true amplitude fully recovered)")
ax.legend(fontsize=8)

ax = axes[1, 0]
ax.plot(frqs_true[mask], spec_true[mask], color="firebrick",
        linewidth=1.2, alpha=0.8, label="True forcing u_true(t)")
ax.plot(frqs_fin[mask], spec_fin[mask], color="steelblue",
        linewidth=1.2, alpha=0.9, label="Estimated u(t;theta) - final")
for f, col in zip(opt.freqs, palette):
    ax.axvline(f, color=col, linestyle=":", linewidth=0.8, alpha=0.5)
ax.set_xlabel("Frequency [Hz]")
ax.set_ylabel("|FFT|")
ax.set_title("Final forcing spectrum: estimated vs true")
ax.legend(fontsize=8)

ax = axes[1, 1]
for idx, col in zip(snap_idx, snap_colors):
    ax.plot(t_eval, res_list[idx][:, 0], color=col, alpha=0.85,
            linewidth=1.0, label=f"iter {iters_m[idx]}")
ax.plot(t_eval, u_true, color="firebrick", linewidth=1.2, alpha=0.7, label="True u(t)")
ax.set_xlabel("t")
ax.set_ylabel("u(t)")
ax.set_title("Forcing - time domain\n(estimated at 3 snapshots vs true)")
ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "misspec_pinn_analysis.png", dpi=150, bbox_inches="tight")
plt.show()